In [0]:
transaction_df = spark.read.table('de_mini_project.azure_blob_storage.transaction')
display(transaction_df)

In [0]:
from pyspark.sql import functions as F
transaction_df = transaction_df.drop("_file","_line","_modified","_fivetran_synced")
transaction_df = transaction_df.withColumnRenamed("RTN_ID_No","return_id").withColumnRenamed("Orgnl_Trxn_ID?","transaction_id").withColumnRenamed("Rtn_Date_String","date").withColumnRenamed("Rsn_Code_!","reason")
#Format Dates
return_df = transaction_df.withColumn("date",F.coalesce(
    F.try_to_date(F.col("date"), "dd-MM-yyyy"),
    F.try_to_date(F.col("date"), "MM-dd-yyyy"),
    F.try_to_date(F.col("date"), "dd-MMM-yy"),
    F.try_to_date(F.col("date"), "yyyy.MM.dd"),
    F.try_to_date(F.col("date"), "MMM dd, yyyy"),
    F.try_to_date(F.col("date"), "yyyyMMdd"),
    F.try_to_date(F.col("date"), "MM/dd/yyyy"),
))
display(return_df)

In [0]:
return_df.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.silver.return")